# Анализ лояльности пользователей Яндекс Афиши

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

- **Хост** — `rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net`
- **База данных** — `data-analyst-afisha`
- **Порт** — `6432`
- **Аутентификация** — `Database Native`
- **Пользователь** — ...
- **Пароль** — ...

Для выгрузки используйте запрос, который вы писали в уроке проекта «Выгрузка данных с помощью SQL». Инструкция по тому, как это сделать, дана в уроке «Подключение к базе данных с помощью Python». Полученные из базы данных результаты сохраните в переменную, чтобы продолжить работу с этими данными далее.

---

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Приступим к выполнению проекта - импортируем библиотеки, создадим подключение к базе данных, получим общую информацию о данных.
    
</div>

In [ ]:
# Импортируем библиотеки
import pandas as pd
import numpy as np

# Библиотеки визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

# Подгрузим phik
from phik import resources, report

# Подключение к базе данных
from sqlalchemy import create_engine

# Создаём подключение -- параметры
db_config = {'user': ..., # имя пользователя
             'pwd': ..., # пароль
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432, # порт подключения
             'db': 'data-analyst-afisha' # название базы данных
             }

# Создаём подключение
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

# Сохраняем коннектор
engine = create_engine(connection_string)

In [ ]:
# Пропишем основной запрос
query = '''

WITH
-- Собираем данные по заказам с мобильных и веб устройств:
orders AS (
    SELECT
        -- Информация о пользователе:
        user_id,
        device_type_canonical,
        -- Информация о заказе:
        order_id, -- Номер заказа
        created_dt_msk AS order_dt, -- Дата заказа
        created_ts_msk AS order_ts, -- Время заказа
        currency_code, -- Код валюты
        revenue, -- Выручка сервиса
        tickets_count, -- Кол-во билетов
        -- Информация о мероприятии:
        event_id,
        service_name,
        -- Найдем время между заказами для каждого пользователя:
        created_dt_msk::date - LAG(created_dt_msk) OVER(PARTITION BY user_id ORDER BY created_dt_msk)::date AS days_since_prev
    FROM afisha.purchases
    WHERE
        -- Фильтруем тип устройства
        device_type_canonical IN ('mobile','desktop')
),
-- Собираем информацию по событиям:
    events AS (
    SELECT
        -- Выгружаем данные таблицы events:
        e.event_id,
        e.event_name_code AS event_name,
        e.event_type_main,
        -- Выгружаем информацию о городе и регионе:
        r.region_name,
        c.city_name
    FROM afisha.events AS e
    LEFT JOIN afisha.venues AS v USING(venue_id)
    LEFT JOIN afisha.city AS c USING(city_id)
    LEFT JOIN afisha.regions AS r USING(region_id)
    WHERE e.event_type_main != 'фильм'
)
-- Создаем итоговую выгрузку:
SELECT
    orders.*,
    event_name,
    event_type_main,
    region_name,
    city_name
FROM orders INNER JOIN events USING (event_id)
ORDER BY user_id, order_dt;

'''

In [ ]:
tickets = pd.read_sql_query(query, con=engine)

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
# Выводим первые строки датафрейма на экран
tickets.head(3)

In [ ]:
# Выводим информацию о датафрейме
tickets.info()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>**Промежуточный вывод**
    
<font color='#253d5f'>Пропуски содержатся в столбце `days_since_prev`, что ожидаемо — для первого заказа пользователя это значение отсутствует. В остальных столбцах пропусков нет. Типы данных определены корректно.

<font color='#253d5f'>Теоретически можно оптимизировать объёмы хранения, понизив разрядность числовых столбцов, но при текущем объёме данных это не так принципиально. В целом можно сказать, что выгрузка выполнена корректно.

<font color='#253d5f'>Однако на этапе предобработки данных рекомендуется:

- <font color='#253d5f'>проверить корректность написания категориальных значений (возможны дубли из-за различий в регистре, опечаток и т.п.);
- <font color='#253d5f'>убедиться, что числовые столбцы не содержат невалидные значения (например, индикаторы пропусков вроде -1, 9999, 0 при ожидании положительных значений);
- <font color='#253d5f'>обратить внимание на возможные аномалии в данных.
    
<font color='#253d5f'>Но, по идее студенты должны были уже познакомиться с данными на стадии SQL, и получить общее представление о данных.

</div>

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно скачать по [ссылке](https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv).

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---

In [ ]:
# Выгружаем данные о курсе валют в переменную tenge
tenge = pd.read_csv('final_tickets_tenge_df.csv')

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Познакомимся с данными датасета `final_tickets_tenge_df.csv`, содержащий информацию о курсе тенге к российскому рублю (100 к 1) за 2024 год:
    
</div>

In [ ]:
# Выводим первые строки датафрейма на экран
tenge.head(3)

In [ ]:
# Выводим информацию о датафрейме
tenge.info()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>По общим данным пропусков в данных нет. Данные соответствуют спецификации. Приступим к созданию нового столбца. На предудыщем уроке студенты уже смотрели на данные с помощью тренажера SQL, поэтому должны знать, что тут только две валюты — российские рубли и казахстанские тенге.
    
</div>

In [ ]:
# Преобразуем дату в tenge и сделаем её индексом
tenge['data'] = pd.to_datetime(tenge['data'])
tenge_map = tenge.set_index('data')['curs']

# Создаем столбец с курсом, используя map
tickets['curs'] = tickets['order_dt'].map(tenge_map)

# Определяем функцию, переводящую выручку с заказа в рубли
def kzt_to_rub(row):
    if row['currency_code'] == 'kzt':
        return row['revenue'] * row['curs'] / 100
    else:
        return row['revenue']

# Создадим столбец с данными о выручке с заказа в рублях
tickets['revenue_rub'] = tickets.apply(kzt_to_rub, axis=1)

In [ ]:
# Проверим результат:
tickets[['revenue','currency_code','revenue_rub']].sample(5)

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Новый стобец создан успешно, можно продолжать занкомство с данными.
    
</div>

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [ ]:
# Проверяем данные на пропуски
tickets.isna().mean()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Пропуски там, где они должны быть. В целом нет необходимости что-то делать. Пропуски в `days_since_prev` оставим как есть, так как это нормальная ситуация, что для первой покупки нет информации по времени от предыдущей покупки. Заменять на нули нельзя, так как могут быть покупки, совершенные в один день.
    
<font color='#253d5f'>Теперь посмотрим на корректность категориальных данных.
    
</div>

In [ ]:
# Проверяем уникальные значения в категориальных столбцах
for column in ['device_type_canonical',
               'service_name',
               'event_name',
               'event_type_main',
               'region_name',
               'city_name'
               ]:
    un_values_count = tickets[column].sort_values().nunique()
    print(f'Уникальные значения в столбце {column}: {un_values_count} значений')
    if un_values_count > 20:
        print(f'Выведем первые 20 значений')
        print(tickets[column].sort_values().unique()[:20])
        print()
    else:
        print(tickets[column].sort_values().unique())
        print()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>В данных, где немного категорий — все нормально, там где категорий много, могут быть опечатки, ошибки, лишние символы. В целом можно их более детально проработать, но не понятно, насколько это нужно.
    
<font color='#253d5f'>Проведем небольшую нормализацию данных: приведем категории, где много значений к нижнему регистру, удалим возможные пробелы в начале и конце строки.

    
</div>

In [ ]:
# Нормализуем данные:
for column in ['device_type_canonical',
               'service_name',
               'event_name',
               'event_type_main',
               'region_name',
               'city_name'
               ]:
    un_values_count = tickets[column].sort_values().nunique()
    if un_values_count > 20:
        tickets[column] = tickets[column].str.lower().str.strip()
    else:
        pass

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Проверять не будем :) поверим коду. Теперь перейдем к числовым данным. Проверим распределение значений в них. Основной акцент сделаем на выручке с билета и количестве билетов в заказе.

    
</div>

In [ ]:
# Изучаем статистические показатели столбца revenue
print('Статистические показатели столбца revenue:')
tickets['revenue_rub'].describe()

In [ ]:
# Создаём контейнер графика matplotlib и задаём его размер
plt.figure(figsize=(14, 4))

# Строим гистограмму с помощью pandas через plot(kind='hist')
tickets['revenue_rub'].plot(
                kind='hist',
                bins=60,
                alpha=0.75,
                edgecolor='black',
                rot=0,
)

# Настраиваем оформление графика
plt.title('Распределение выручки с заказа в рублях')
plt.xlabel('Выручка с заказа в рублях')
plt.ylabel('Частота')

plt.yscale('log')

plt.grid()
plt.show()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Можно обратить внимание на большой разброс значений. Плюс есть отрицательные значения — может быть это возвраты билетов. Такие уберём данные, так как они могут завысить кол-во транзакций на пользователя. Также отфильтруем данные по 99 перцентилю.

    
</div>

In [ ]:
# Фильтруем выручку в рублях по 99 перцентилю:
rub99 = np.percentile(tickets['revenue_rub'], 99)
print(f'Значение 99 перцентиля для заказов в рублях: {rub99:.0f}')

df = tickets[(tickets['revenue_rub'] < rub99) & (tickets['revenue_rub'] > 0)]

In [ ]:
# Изучаем статистические показатели столбца revenue
print('Статистические показатели столбца revenue:')
df['revenue_rub'].describe()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Данные были почищены от выбросов и аномалий. Проверим количество билетов.

    
</div>

In [ ]:
# Проверим статистику теперь:
df.groupby('currency_code').agg(
    tickets_min = ('tickets_count', 'min'),
    tickets_max = ('tickets_count', 'max'),
    tickets_avg = ('tickets_count', 'mean'),
    tickets_p95 = ('tickets_count', lambda x: np.percentile(x, 95)),
    tickets_p99 = ('tickets_count', lambda x: np.percentile(x, 99)),
    tickets_var = ('tickets_count', 'var'))

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Есть заказы с большим количеством билетов, но тут нет ничего необычного. В целом 50 билетов можно считать за выброс, но кажется, что такие заказы могут быть, например, для какой-то группы или экскурсии. Фильтровать данные не будем. Пойдем дальше.
    
</div>

In [ ]:
# Проверяем наличие дубликатов
df.duplicated().sum()

In [ ]:
# Проверяем наличие неявных дубликатов - испльзуем время покупки
columns_to_dupl = ['user_id', 'order_ts', 'device_type_canonical',
    'service_name', 'event_id', 'revenue']
df[columns_to_dupl].duplicated().sum()

In [ ]:
# Выводим неявных дубликаты на экран
df[df[columns_to_dupl].duplicated(keep=False)].head(6)

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Мы видим полные дубликаты строк, хотя номера заказов разные. Но их не так много, поэтому убедимся, что доля малая и удалим.
    
</div>

In [ ]:
# Выводим неявные дубликаты на экран:
df[columns_to_dupl].duplicated().sum() / len(df)

In [ ]:
# Удаляем неявные дубликаты
df = df.drop_duplicates(subset=columns_to_dupl)

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Неявных дубликатов было мало, но мы их удалили и хорошо. Типы данных принципиально нет необходимости корректировать, но, для примера того, что могут сделать студенты, понизим размерность данных.
    
</div>

In [ ]:
# Уменьшим размерность данных
for column in ['order_id','event_id','tickets_count']:
    df[column] = pd.to_numeric(df[column], downcast='integer')

for column in ['revenue','revenue_rub','days_since_prev']:
    df[column] = pd.to_numeric(df[column], downcast='float')

In [ ]:
# Проверяем результат
df.info()

In [ ]:
# Подсчитаем процент отфильтрованных данных:
print(f'Было отфильтровано {1 - len(df) / len(tickets):.2%} данных')

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>**Промежуточный вывод по предобработке данных**

<font color='#253d5f'>Была проведена преобработка данных:

- <font color='#253d5f'>Пропусков в данных, которые могли быть ошибкой нет.
- <font color='#253d5f'>Были изучены категориальные и числовые данные на корректность — явных ошибок обнаружено не было. Строковые данные с большим количеством уникальных значений были нормализованы. Выручка с заказа в рублях была отфильтрована по **99 перцентилю**.
- <font color='#253d5f'>В данных были обнаружены неявные дубликаты в заказе — полные дубли, за исключением номера заказа. Такие данные были отфильтрованы. Но их доля была невысокой.
- <font color='#253d5f'>Был создан новый столбец: `revenue_rub` — выручка в рублях;
- <font color='#253d5f'>Типы данных были оптимизированы.
    
<font color='#253d5f'>После предобработки было отфильтровано примерно **3 процента данных**.
    
</div>

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее время между заказами.

После этого добавьте бинарный признак:

- `is_two` — совершил ли пользователь больше двух заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Начнем собирать профиль пользователя для дальнейшей работы.
</div>

In [ ]:
# Собираем профиль пользователя:
profiles = (df
            # В начале сортируем данные по дате совершения заказа, что найти первые признаки:
            .sort_values(by='order_ts')
            # Затем группируем по номеру пользователя и агрегируем данные:
            .groupby('user_id')
            .agg(
                # Находим первую и последнюю даты заказа:
                first_order_dt=('order_dt','min'),
                last_order_dt=('order_dt','max'),
                # Находим устройства и регион первого заказа:
                first_device=('device_type_canonical','first'),
                first_region_name=('region_name','first'),
                # Жанр первого посещённого мероприятия (event_type_main):
                first_event_type=('event_type_main','first'),
                # Посчитаем количество заказов:
                total_orders=('order_id','nunique'),
                # Посчитаем среднюю стоимость заказа:
                avg_revenue_rub=('revenue_rub','mean'),
                # Считаем среднее количество дней между покупками:
                avg_days_since_prev=('days_since_prev','mean')
            )
            # Создаем бинарный признак: совершил ли пользователь больше 2 заказов:
            .assign(
                is_two = lambda x: x['total_orders'] >= 2
            )
            # Можно альтернативным образом подсчитать среднее количество дней между заказами (если не будет в SQL):
            .assign(
                avg_days = lambda x: (x['last_order_dt'] - x['first_order_dt']).dt.days / (x['total_orders'] - 1)
            )
            .reset_index()
)

# Проверяем результат:
profiles.head()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>В целом что-то похожее студенты уже делали в предыдущем модуле, но с помощью SQL. По идее, тут все просто. Для примера показано, как студенты могут рассчитать среднее время между заказами, если не считали дату предыдущего заказа с использованием SQL.
    
<font color='#253d5f'>Тут самое важное, чтобы в начале была проведена сортировка по дате заказа, чтобы первые значения подцепить.
</div>

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших более 2 заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

---

In [ ]:
# Изучим общее количество пользователей в полученных данных:
print(f"Общее число пользователей: {profiles['user_id'].nunique()}")

# Подсчитаем долю пользователей, совершивших 2 и более заказа:
print(f"Доля пользователей совершивших 2 и более заказа: {profiles['is_two'].mean():.2f}")

# Изучим статистику по общему числу заказов, среднему числу билетов в заказе и среднему количеству дней между покупками:
print(f"""
Статистические показатели по:
- общему числу заказов - total_orders,
- средней выручки с одного заказа - avg_revenue_rub,
- среднему количеству дней между покупками - avg_days_since_prev.""")

profiles[['total_orders','avg_revenue_rub', 'avg_days_since_prev']].describe(percentiles=[.01, .25, .5, .75, .95, .99]).T

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>В целом количество пользователей достаточно высокое - `21694`, можно провести анализ полученных результатов.

<font color='#253d5f'>Также данные показывают, что есть отрицательные значения по выручке — возможно это возврат билетов, таких пользователей мы исключим, так как сейчас стоят другие задачи для анализа.
    
<font color='#253d5f'>Отфильтруем пользователей по количеству заказов выше `99` перцентиля, уберем отрицательные значения и еще раз посмотрим на статистику:
    
</div>

In [ ]:
# Фильтруем выручку в рублях по 99 перцентилю:
orders99 = np.percentile(profiles['total_orders'], 99)
print(f'Значение 99 перцентиля для общего количества заказов для фильтрации: {orders99:.0f}')
profiles_f = profiles[(profiles['total_orders'] < orders99) & (profiles['avg_revenue_rub'] > 0)]

# Изучим общее количество пользователей в полученных данных:
print(f"Общее число пользователей: {profiles_f['user_id'].nunique()}")
print(f"Было отфильтровано: {1-profiles_f['user_id'].nunique()/profiles['user_id'].nunique():.2%} пользователей")

# Подсчитаем долю пользователей, совершивших 2 и более заказа:
print(f"Доля пользователей совершивших 2 и более заказа: {profiles['is_two'].mean():.2f}")

# Изучим статистику по общему числу заказов, среднему числу билетов в заказе и среднему количеству дней между покупками:
print(f"""
Статистические показатели по:
- общему числу заказов - total_orders,
- средней выручки с одного заказа - avg_revenue_rub,
- среднему количеству дней между покупками - avg_days_since_prev.""")

profiles_f[['total_orders','avg_revenue_rub', 'avg_days_since_prev']].describe(percentiles=[.01, .25, .5, .75, .95, .99]).T

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Данные немного усреднились, но общий объём сохранился. Будем работать с ними. В результате была сформулирована одна целевая переменная:
    
1. <font color='#253d5f'>**Целевая переменная** — пользователь совершил повторную покупку:
    - <font color='#253d5f'>`1` – пользователь совершил 2+ покупок (повторный покупатель).
    - <font color='#253d5f'>`0` – пользователь совершил только 1 покупку.

    
</div>

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.

#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---

In [ ]:
# Проведем необходимый анализ: в начале выделим первые признаки:
segments = ['first_device','first_event_type','first_region_name']

# Для каждого сегмента найдем необходимые значения:
for segment in segments:
    print(f'Распределение пользователей по признаку {segment}')
    display(profiles
             .groupby(segment).agg(total_users=('user_id','nunique'))
             .assign(users_share=lambda x: x['total_users'] / x['total_users'].sum())
             .sort_values(by='users_share', ascending=False)
           )

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Результаты показывают:
- <font color='#253d5f'>Большая часть пользователей использовали в качестве первого устройства, с которого был совершен заказ, мобильные устройства - это сделало практически `83%` пользователей.
- <font color='#253d5f'>`44%` пользователей в качестве первого заказа купили билеты на концерты, затем идут такие мероприятия, как театры (почти `20%`), остальные категории — стендапы, спортивные мероприятия, выставки и елки — менее популярны (`от 5 до 0.5%`). Однако в данных есть еще одна категория мероприятия — укрупненная группа другое. В эту группу попало около `25%` пользователей.
- <font color='#253d5f'>Количество регионов высокое — всего `81 регион`. Пользователи распределены крайне неравномерно — есть два региона лидеров (`32` и `17%` — по количеству не трудно догадаться, что это Москва и Санкт-Петербург — почти половина выборки). Доля пользователей из других регионов `не выше 6%`. Есть регионы, представленные единичными пользователями. При исследовании доли пользователей с повторными покупками будем изучать `топ-10 регионов`.

    
</div>

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментом по региону.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---

In [ ]:
# Изучим доли покупателей с 2 и более покупками:
# Для каждого сегмента найдем необходимые значения:
for segment in segments:
    print(f'Распределение пользователей по признаку {segment}')
    # Считаем доли вернувшихся пользователей:
    res = (profiles
             .groupby(segment).agg(
                 total_users=('user_id','nunique'),
                 is_two_share=('is_two','mean'))
             .nlargest(10, 'total_users')
             .sort_values(by='is_two_share', ascending=False)
           )

    # Смотрим, является ли значение выше среднего или нет:
    res['is_higher_mean'] = res['is_two_share'] > profiles['is_two'].mean()

    # Проверяем, какой охват выборки:
    share = res["total_users"].sum()/len(profiles)
    print(f'Выборка охватывает {share:.1%} пользователей')

    # Выводим результат:
    display(res)

In [ ]:
# Построим график столбчатой диаграммы
for segment in segments:
    # Группируем данные по признакам:
    grouped = (profiles
               .groupby(segment)
               .agg(total_users=('is_two','count'),
                    is_two_share=('is_two','mean'))
               .nlargest(10, 'total_users')
               .sort_values(by='is_two_share', ascending=False)
               .head(10))

    # Визуализируем результаты (с контролем позиции меток):
    rot_value = 45 if len(grouped.index) > 7 else 0
    grouped[['is_two_share']].plot(kind='bar',
                   title=f'Распределение доли пользователей с двумя и более заказами \nв зависимости от значения признака {segment}',
                   legend=True,
                   ylabel='Доля вернувшихся клиентов',
                   xlabel=f'Сегмент признака {segment}',
                   rot=rot_value,
                   figsize=(10, 4))
    plt.grid()

    # Рассчитываем среднее значение по доле нелояльных клиентов
    mean_share = profiles['is_two'].mean()

    # Наносим на график линию с средним значением доли нелояльных клиентов
    plt.axhline(mean_share,
                color='red',
                linestyle='--',
                linewidth=1,
                label=f'Доля вернувшихся клиентов по всем данным - {mean_share:.3f}')

    # Скорректируем легенду:
    plt.legend(loc='lower center',
               ncol=2)

    # Выводим график
    plt.show()

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Рассмотрим результаты анализа по каждому фактору:
- <font color='#253d5f'>Пользователи, сделавшие первый заказ с **десктопных устройств**, показали наивысшую долю возвращаемости — 64.4% против 61.2% у мобильных. Таким образом, можно предположить, что десктопное использование ассоциируется с более высоким вовлечением. Хотя не ясно, является ли разница значимой или нет.
- <font color='#253d5f'>Наиболее перспективные категории для первого взаимодействия — **выставки, театр и концерты**: все они показывают долю возвращаемости выше среднего. Наименьшие значения зафиксированы у **спорта и ёлок**. Гипотеза о том, что тип мероприятия влияет на возвращаемость, подтверждается: однако результат **протиповоложный** — культурные мероприятия вовлекают лучше, чем спорт.   
- <font color='#253d5f'>Для анализа использованы 10 регионов с наибольшим числом пользователей (охват — 72%). Из них шесть регионов демонстрируют долю возвращаемости выше среднего. При этом размер региона (по числу пользователей) не всегда соответствует высокой доле удержания — что говорит о наличии других факторов вовлечения и вторая предложенная гипотеза не подтверждается.
    
<font color='#253d5f'>Первые признаки взаимодействия (устройство, мероприятие, регион) действительно могут быть связаны с вероятностью возврата пользователей. Особенно заметно влияние типа мероприятия и устройства, а также отдельных регионов.
    
</div>

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---

In [ ]:
# Строим гистограмму распределения значений возраста
# Создаём фигуру графика
plt.figure(figsize=(10, 4))

# Находим минимальное и максимальное значения
min_value = int(profiles['avg_revenue_rub'].min())
max_value = round(profiles['avg_revenue_rub'].max())

# Строим гистограммы для каждого значения churn
for i in profiles['is_two'].unique():
    # Фильтруем данные по значению столбца churn
    profiles.loc[profiles['is_two'] == i, 'avg_revenue_rub'].plot(
        kind='hist',
        density=True,
        bins=range(min_value, max_value+1, 50),
        alpha=0.5,
        label=f'{i}',
        legend=True
    )

# Настраиваем внешний вид графика и выводим его на экран
plt.title(f'Сравнение распределения средней выручки с заказа')
plt.xlabel('Средняя выручка с билета')
plt.ylabel('Плотность вероятности')
plt.legend(title='Значение признака is_two')

plt.grid()
plt.show()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Общий вывод по результатам — пользователи, которые возвращаются, в среднем приносят больше выручки с одного билета, чем те, кто сделал один заказ:
- <font color='#253d5f'>Для пользователей с одним заказом распределение резко смещено влево. Пик около 100–200 руб, затем резкий спад.
- <font color='#253d5f'>Для  возвращающихся пользователей распределение сдвинуто правее, пик ближе к 500–600 руб. Более плавное распределение, без резких скачков.
    
<font color='#253d5f'>Обе группы имеют длинные "хвосты" справа, уходящие за 1000 руб, в редких случаях — даже за 3000 руб.
    
<font color='#253d5f'>Гипотеза о том, что пользователи, чья выручка с заказа высокая, т.е. они оформляют более дорогие заказы, чаще совершают повторные покупки, подтверждается.
    
</div>

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---

In [ ]:
# Выделяем день недели из даты заказа
profiles['weekday'] = pd.to_datetime(profiles['first_order_dt']).dt.day_name()

In [ ]:
# Группируем по сегментам и считаем долю пользователей с повторными заказами
res = (profiles
        .groupby('weekday')
        .agg(total_users=('is_two', 'count'),
             is_two_share=('is_two', 'mean'))
      )
res = res.reindex(['Monday','Tuesday','Wednesday','Thursday','Friday',
                  'Saturday', 'Sunday'])

# Визуализируем результаты (с контролем позиции меток):
res[['is_two_share']].plot(kind='bar',
                   title=f'Распределение доли пользователей с двумя и более заказами \nв зависимости от значения признака {segment}',
                   legend=True,
                   ylabel='Доля вернувшихся клиентов',
                   xlabel=f'Сегмент признака {segment}',
                   rot=0,
                   figsize=(10, 4))
plt.grid()

# Рассчитываем среднее значение по доле нелояльных клиентов
mean_share = profiles['is_two'].mean()

# Наносим на график линию с средним значением доли нелояльных клиентов
plt.axhline(mean_share,
                color='red',
                linestyle='--',
                linewidth=1,
                label=f'Доля вернувшихся клиентов по всем данным - {mean_share:.3f}')

    # Скорректируем легенду:
plt.legend()

# Выводим график
plt.show()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Распределение пользователей по дням недели первой покупки и вероятность повторных заказов:
    
- <font color='#253d5f'>Общее количество пользователей, совершивших первую покупку в разные дни недели, примерно равномерно распределено, с небольшими колебаниями от 2800 до 3500 человек в каждый день.
    
- <font color='#253d5f'>Доля пользователей, совершивших повторные заказы (`is_two_share`), колеблется от 59.5% до 64.2%.
    
- <font color='#253d5f'>Наивысшая вероятность повторных покупок наблюдается у пользователей, сделавших первую покупку в субботу (64.2%) и в понедельник (63.3%).
    
- <font color='#253d5f'>Наименьшая доля повторных покупок у пользователей с первой покупкой в четверг (59.5%) и пятницу (59.8%).
    
<font color='#253d5f'>Общий вывод: различия в долях не очень велики, но указывает на некоторую зависимость между днем первой покупки и вероятностью возвращения.
    
</div>

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - 2 заказа;
    - от 3 до 4 заказов (оба значения включительно);
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
# Вычисляем корреляционную матрицу с использованием phi_k
correlation_matrix = profiles[['first_device', 'first_region_name', 'first_event_type', 'weekday', 
                               'avg_revenue_rub', 'avg_days_since_prev', 
                               'avg_days', 'total_orders']].dropna().phik_matrix(
    interval_cols=['avg_revenue_rub', 'avg_days_since_prev', 'avg_days', 'total_orders']
)

# Выводим результат
print('Корреляционная матрица с коэффициентом phi_k для переменной total_orders')
correlation_matrix.loc[correlation_matrix.index != 'total_orders'][['total_orders']].sort_values(by='total_orders', ascending=False)

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Видим много нулей, проверим распределение `total_orders`
    
</div>

In [ ]:
# Проверим распределение данных по кол-ву покупок
profiles['total_orders'].describe()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Распределение данных показывает, что много низких значений. Поэтому лучше провести корреляционный анализ, выделив группы пользователей по покупкам:
</div>

In [ ]:
# Создаем функцию для сегментации по среднему количеству покупок
def segment_orders(x):
    if 1 <= x < 2:
        return 'от 1 до 2 покупок'
    elif 2 <= x < 3:
        return 'от 2 до 3 покупок'
    elif 3 <= x < 5:
        return 'от 3 до 5 покупок'
    else:
        return 'от 5 и более покупок'

# Добавляем столбец с сегментами
profiles['orders_segment'] = profiles['total_orders'].apply(segment_orders)

profiles['orders_segment'].value_counts()

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Сегменты получаются примерно равные. Проверим их корреляцию:
</div>

In [ ]:
# Вычисляем корреляционную матрицу с использованием phi_k
correlation_matrix = profiles[['first_device', 'first_region_name', 'first_event_type', 'weekday', 
                               'avg_revenue_rub', 'avg_days_since_prev', 'avg_days', 'orders_segment']].phik_matrix(
    interval_cols=['avg_revenue_rub', 'avg_days_since_prev', 'avg_days']
)

# Выводим результат
print('Корреляционная матрица с коэффициентом phi_k для переменной orders_segment')
correlation_matrix.loc[correlation_matrix.index != 'orders_segment'][['orders_segment']].sort_values(by='orders_segment', ascending=False)

<div class="alert"
     style="background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px; border: 1px solid #253d5f;
			color: #253d5f;">

<font color='#253d5f'>Другое дело. Наиболее сильная связь с сегментами по количеству покупок (`orders_segment`) наблюдается у среднего времени между покупками (`avg_days_since_prev`). Также заметна умеренная связь со средней выручкой (`avg_revenue_rub`).
    
<font color='#253d5f'>Категориальные признаки, такие как регион или устройство, показывают слабую корреляцию. Это говорит о том, что поведение клиентов во времени важнее для объяснения повторных покупок, чем их демография или устройство.
    
<font color='#253d5f'>Теперь результаты в целом согласуются с теми результатами, которые мы получили при анализа данных выше.
</div>

In [ ]:
orders_segment = correlation_matrix.loc[correlation_matrix.index != 'orders_segment'][['orders_segment']].sort_values(by='orders_segment', ascending=False)

plt.figure(figsize=(2, 5))
sns.heatmap(orders_segment, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5, cbar=False)
plt.title('Тепловая карта phi_k корреляции с orders_segment')
plt.xlabel('Общее число заказов')
plt.show()

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.